# DEG → Functional Enrichment

Pre-ranked GSEA on per-(sex × cell type) DEG CSVs followed by a side-by-side pathway heatmap.

All logic lives in `pygenelab.deg_functional_enrichment`. The DEG dict in **Cell A** drives everything — any number of cell types, either gender, any DEG source as long as the CSV has a gene column + signed log-fold-change + adjusted p-value.

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

PYGENELAB_PARENT = Path("/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/TA_muscle_code/py_scripts")
if str(PYGENELAB_PARENT) not in sys.path:
    sys.path.insert(0, str(PYGENELAB_PARENT))

import pygenelab as pgl
import pandas as pd
import matplotlib.pyplot as plt

## Cell A — pick the DEG files

`DEG_FILES` maps a label (used as the heatmap column) to a CSV path. Any sex / any cell type / any number of entries.

Each CSV must have:
- a gene column (the first column is auto-renamed to `GENE_COL`)
- a signed log-fold-change column (`LOG2FC_COL`)
- an adjusted p-value column (`PVAL_COL`)

In [ ]:
# --- Female Fast IIX vs IIB (Seurat output, CRC paths) ---
DEG_FILES = {
    "IIX": "/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/Seurat_analysis_outputs/tables/Female_FastIIX_unfiltered_KO_DEGs.csv",
    "IIB": "/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/Seurat_analysis_outputs/tables/Female_FastIIB_unfiltered_KO_DEGs.csv",
}
ANALYSIS_LABEL = "Female_FastIIX_vs_FastIIB"
# Seurat column conventions
GENE_COL = "gene_name"
LOG2FC_COL = "avg_log2FC"
PVAL_COL = "p_val_adj"

# --- Male Fast IIX vs IIB (Seurat output, CRC paths) ---
# DEG_FILES = {
#     "IIX": "/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/Seurat_analysis_outputs/tables/Male_Fast IIX_unfiltered_KO_DEGs.csv",
#     "IIB": "/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/Seurat_analysis_outputs/tables/Male_Fast IIB_unfiltered_KO_DEGs.csv",
# }
# ANALYSIS_LABEL = "Male_FastIIX_vs_FastIIB"
# GENE_COL = "gene_name"
# LOG2FC_COL = "avg_log2FC"
# PVAL_COL = "p_val_adj"

# --- PSC scanpy-style DEGs (column names differ) ---
# DEG_FILES = {
#     "IIX": "/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/analysis/male/degs/DEGs_KO_vs_WT_Fast_IIX.csv",
#     "IIB": "/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/analysis/male/degs/DEGs_KO_vs_WT_Fast_IIB.csv",
# }
# ANALYSIS_LABEL = "Male_FastIIX_vs_FastIIB_scanpy"
# GENE_COL = "gene_name"
# LOG2FC_COL = "logfoldchanges"
# PVAL_COL = "pvals_adj"

for label, p in DEG_FILES.items():
    print(f"  {label:>20s}: {p}")

## Cell B — pathway source

Pick the GMT and (optionally) keywords to restrict pathway names. The size filter keeps pathways with 15–500 genes by default.

In [ ]:
# --- Mouse MSigDB (full) ---
PATHWAYS_GMT = "/ix/djishnu/Akanksha/datasets/gene_sets/mouse/msigdb.v2024.1.Mm.symbols.gmt"
# PSC: /ocean/projects/cis240075p/asachan/datasets/gene_sets/mouse/msigdb.v2024.1.Mm.symbols.gmt
GENE_ORIGIN = "mice"

PATHWAY_TYPE = "METABOLISM"
PATHWAY_KEYWORDS = ["METABOLISM"]
# PATHWAY_KEYWORDS = ["DNA_REPAIR", "DNA_DAMAGE"]
# PATHWAY_KEYWORDS = None  # keep all pathways within the size band

PATHWAY_SIZE_MIN = 15
PATHWAY_SIZE_MAX = 500

## Output directory

In [ ]:
OUTPUT_DIR = Path(
    "/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/TA_muscle_code/py_scripts/3_geneset_scores/Output/Enriched_Pathways"
) / ANALYSIS_LABEL / PATHWAY_TYPE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUTPUT_DIR)

## Step 1 — load DEG CSVs

In [ ]:
degs = pgl.load_deg_csvs(DEG_FILES, gene_col=GENE_COL)
for label, df in degs.items():
    print(f"  {label}: {df.shape[0]} genes, columns = {list(df.columns)}")

## Step 2 — pre-ranked gene lists per DEG file

Ranking = signed log-fold-change × −log10(adjusted p-value).

In [ ]:
ranked_lists = {
    label: pgl.create_ranked_genelist(
        df,
        log2fc_col=LOG2FC_COL,
        pval_col=PVAL_COL,
        gene_col=GENE_COL,
    )
    for label, df in degs.items()
}
for label, r in ranked_lists.items():
    print(f"  {label}: ranked shape = {r.shape}")

## Step 3 — load + filter pathways

In [ ]:
pathways_all = pgl.convert_gmt_to_decoupler_format(PATHWAYS_GMT, gene_origin=GENE_ORIGIN)
pathways = pgl.filter_pathways(
    pathways_all,
    size_min=PATHWAY_SIZE_MIN,
    size_max=PATHWAY_SIZE_MAX,
    keywords=PATHWAY_KEYWORDS,
)
print(f"  pathways retained: {pathways['source'].nunique()}")

## Step 4 — GSEA per DEG file

In [ ]:
gsea_results = {label: pgl.run_gsea(r, pathways) for label, r in ranked_lists.items()}
for label, res in gsea_results.items():
    print(f"{label}: {res.shape[0]} pathways scored")
    display(res.head(3))

## Step 5 — annotate + dedup pathway terms

In [ ]:
plot_dfs = {
    label: pgl.dedup_by_simplified_term(pgl.prepare_gsea_plot_df(res))
    for label, res in gsea_results.items()
}
for label, df in plot_dfs.items():
    print(f"  {label}: {df.shape[0]} unique simplified terms")

## Step 6 — common pathways + heatmap

Pathways shared across every label in `DEG_FILES` get one row; columns follow the dict's insertion order.

In [ ]:
shared = pgl.common_pathways(plot_dfs)
print(f"  shared pathways: {len(shared)}")

heatmap_fig, panels = pgl.plot_pathway_heatmap(
    plot_dfs,
    pathways=shared,
    title=f"{ANALYSIS_LABEL} \u2014 {PATHWAY_TYPE}",
)
heatmap_fig.savefig(OUTPUT_DIR / f"{ANALYSIS_LABEL}_{PATHWAY_TYPE}.png", dpi=300, bbox_inches="tight")
heatmap_fig.savefig(OUTPUT_DIR / f"{ANALYSIS_LABEL}_{PATHWAY_TYPE}.svg", bbox_inches="tight")
heatmap_fig

## (optional) Run everything in one call

In [ ]:
# result = pgl.run_pipeline(
#     deg_paths=DEG_FILES,
#     pathways_gmt=PATHWAYS_GMT,
#     gene_origin=GENE_ORIGIN,
#     pathway_size_min=PATHWAY_SIZE_MIN,
#     pathway_size_max=PATHWAY_SIZE_MAX,
#     pathway_keywords=PATHWAY_KEYWORDS,
#     log2fc_col=LOG2FC_COL,
#     pval_col=PVAL_COL,
#     gene_col=GENE_COL,
#     title=f"{ANALYSIS_LABEL} \u2014 {PATHWAY_TYPE}",
#     output_dir=OUTPUT_DIR,
#     save_basename=f"{ANALYSIS_LABEL}_{PATHWAY_TYPE}",
# )
# result.keys()